# 1. Load S3 data and create model features

We use 15-minute bars from one year of BTCUSDT candles stored in S3. The target is DOWN, FLAT, or UP 60 minutes later. FLAT means the 60-minute move is smaller than 0.20% of the current price.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "s3_to_dataframe.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from s3_to_dataframe import load_data_from_s3, prepare_dataframe
from feature_engineering import FEATURE_COLUMNS, aggregate_candles, create_features

In [ ]:
# Download the same CSV that is stored in S3 and keep a local copy for inspection.
raw_df = prepare_dataframe(load_data_from_s3())
data_dir = ROOT / "data"
data_dir.mkdir(exist_ok=True)
raw_df.to_csv(data_dir / "historical_btcusdt_15m_1y.csv", index=False)
print(f"Raw rows: {len(raw_df):,}")
print(f"Saved local view: {data_dir / 'historical_btcusdt_15m_1y.csv'}")
display(raw_df.head())

In [ ]:
bars = aggregate_candles(raw_df, timeframe="15min")
feature_df = create_features(raw_df, timeframe="15min", horizon_bars=4, flat_threshold=0.002)
print(f"15-minute bars: {len(bars):,}")
print(f"Usable feature rows: {len(feature_df):,}")
print(f"Feature columns: {FEATURE_COLUMNS}")
display(feature_df[["open_time_utc", "close", *FEATURE_COLUMNS, "target"]].tail())

Features use only the current bar and trailing history: multi-window returns, EMA trend, RSI, rolling volatility, relative volume, candle range/position, taker-buy volume, and cyclical time-of-day. The future close is used only to create `target`, never as an input feature.

In [ ]:
display(feature_df["target"].value_counts(normalize=True).rename({-1: "DOWN", 0: "FLAT", 1: "UP"}))
feature_df.to_csv(data_dir / "features_btcusdt_15min_60min.csv", index=False)
print(f"Saved feature view: {data_dir / 'features_btcusdt_15min_60min.csv'}")